# ml4seeding Package Orientation

This notebook demonstrates the core capabilities of the `ml4seeding` package using **synthetic data** — no real drone imagery or TensorFlow required.

**ml4seeding** provides tools for microsite classification from high-resolution drone imagery to support precision aerial reforestation:

- **Orthomosaic analysis**: metadata extraction, quality assessment, vegetation indices
- **Pseudo-orthomosaic generation**: GPS-based stitching of raw drone images
- **Tiling**: fixed-size tile generation with georeferenced metadata
- **Grid overlay**: camouflage grid for annotation quality control
- **Data preprocessing**: image-mask matching, spatial splitting, filtering, oversampling
- **Evaluation metrics**: per-class IoU, mean IoU, foreground mean IoU
- **U-Net model**: semantic segmentation architecture (requires TensorFlow)

In [ ]:
import ml4seeding

print(f"ml4seeding version: {ml4seeding.__version__}")

## 1. Orthomosaic Analysis

The `orthomosaic` module provides quality assessment tools for drone orthomosaics.
Let's create a synthetic RGB image and compute vegetation indices.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml4seeding.orthomosaic import edge_density, laplacian_variance, rgb_indices

# Create a synthetic RGB image (simulating a drone orthomosaic)
np.random.seed(42)
img = np.zeros((100, 100, 3), dtype=np.uint8)
img[:50, :, 1] = 200  # Top half: high green (vegetation)
img[:50, :, 0] = 50
img[:50, :, 2] = 30
img[50:, :, 0] = 150  # Bottom half: brownish (soil/dead vegetation)
img[50:, :, 1] = 100
img[50:, :, 2] = 50

# Add some noise for realism
noise = np.random.randint(-20, 20, img.shape, dtype=np.int16)
img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

# Compute vegetation indices
exg, vari, grvi = rgb_indices(img)

# Compute sharpness and edge density
gray = (0.2989 * img[..., 0] + 0.5870 * img[..., 1] + 0.1140 * img[..., 2])
print(f"Laplacian variance (sharpness): {laplacian_variance(gray):.1f}")
print(f"Edge density: {edge_density(gray):.4f}")

In [ ]:
# Visualize the image and vegetation indices
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img)
axes[0].set_title("RGB Image")
axes[0].axis("off")

im1 = axes[1].imshow(exg, cmap="RdYlGn")
axes[1].set_title("ExG (Excess Green)")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(vari, cmap="RdYlGn")
axes[2].set_title("VARI")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2])

im3 = axes[3].imshow(grvi, cmap="RdYlGn")
axes[3].set_title("GRVI")
axes[3].axis("off")
plt.colorbar(im3, ax=axes[3])

plt.tight_layout()
plt.show()

## 2. Tiling

In [ ]:
from ml4seeding.tiling import compute_valid_bbox, invalid_fraction

# Simulate a tile with some invalid (black) pixels
tile = img.copy()
tile[:, :20] = 0  # Left 20% is black (invalid)

print(f"Invalid fraction: {invalid_fraction(tile):.2f}")
print(f"Valid bbox: {compute_valid_bbox(tile)}")

## 3. Grid Overlay

In [ ]:
from ml4seeding.grid import draw_grid_camo, grid_spacing_px

# Add a camouflage grid to the tile
spacing = grid_spacing_px(pixel_size_m=0.0075, grid_cm=10.0)
print(f"Grid spacing: {spacing} pixels (for 10 cm at 0.75 cm/px)")

gridded = draw_grid_camo(img, spacing=spacing)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img)
axes[0].set_title("Original Tile")
axes[0].axis("off")
axes[1].imshow(gridded)
axes[1].set_title("With Camouflage Grid (10 cm)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 4. Evaluation Metrics

In [ ]:
from ml4seeding.metrics import evaluate_segmentation

# Simulate ground truth and predictions
y_true = np.array([[0, 0, 1, 1], [0, 0, 1, 1], [2, 2, 3, 3], [2, 2, 3, 3]])
y_pred = np.array([[0, 0, 1, 1], [0, 1, 1, 1], [2, 2, 3, 3], [2, 3, 3, 3]])

result = evaluate_segmentation(y_true, y_pred)
print("Evaluation Results:")
for k, v in result.items():
    print(f"  {k}: {v:.4f}")

## 5. U-Net Architecture

In [ ]:
from ml4seeding.models import ARCHITECTURE

print("U-Net Architecture:")
for k, v in ARCHITECTURE.items():
    print(f"  {k}: {v}")

print("\nNote: Model building requires TensorFlow (pip install ml4seeding[ml])")

## Next Steps

- **CLI**: Try `ml4seeding --help` for command-line tools
- **Real data**: See the data management guide for accessing drone imagery
- **Training**: Use `ml4seeding[ml]` with TensorFlow for model training
- **API docs**: See the full API reference in the Sphinx documentation